<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.6-memory-hitl-mcp/notebooks/GCP_Capstone_8.6_Memory.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.6 Integrate Memory, Human-in-the-Loop & MCP Tools — On the Lane, With the Deployed Server
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

8.5 made a conversation survive a restart. It still starts every new conversation from nothing, calls every tool without asking, and cannot reach the MCP server 7.2 deployed. This notebook adds the three things that change that, over the kit's adapters:

- a **store**, keyed to the user rather than the thread - and the silent failure that makes one return everything it has
- a **PII gate** and an **audit row** on every memory write and delete
- **`interrupt()`** to stop and ask before the agent leaves DocuMind's corpus
- Module 7's **deployed MCP server**, reached with a credential minted per request, and *called*

*Prerequisites: 8.5 (this extends its graph) and 7.2 (the server). API facts verified 2026-09-04 against langgraph 1.2.11 and langchain-mcp-adapters 0.3.2.*


## Setup
7.1's, plus the MCP URL. Two kinds of state: the checkpointer holds this conversation; the store holds this user.


In [ ]:
!pip install -q "langgraph==1.2.11" "langgraph-checkpoint-sqlite==3.1.1" "langgraph-checkpoint-postgres==3.1.2" \
                "psycopg[binary]==3.3.5" "langchain-mcp-adapters==0.3.2" "langchain-google-genai==4.4.0" \
                google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every agent in this module imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": f"https://documind-api-{NUMBER}.{REGION}.run.app",
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MCP_URL = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # 7.2's server, for Cell 8
USER_ID = "priya"

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - 8.7's gate fails a paste.
# Two kinds of state, and the difference is the whole lesson:
#   CHECKPOINTER  -> this conversation. Keyed by thread_id. Dies when the thread does.  (8.5)
#   STORE         -> what you know about this USER, across every conversation they have had.
# 8.5 gave DocuMind the first. A checkpointer alone means every new chat starts from nothing,
# which users read as "it forgot me" even though each individual thread resumes perfectly.
print("kit:", KIT, "| API:", os.environ["RAG_API_URL"])


## Cell 1: The namespace is the isolation boundary


In [ ]:
from langgraph.store.memory import InMemoryStore

# Namespaces are TUPLES, and the tuple IS the isolation boundary.
#     (tenant, user, "memories")
# Search takes a namespace PREFIX, so ("acme",) sees every acme namespace and nothing else.
store = InMemoryStore()

NS = (TENANT, USER_ID, "memories")
store.put(NS, "m1", {"text": "asked about leave encashment limits", "kind": "topic"})
store.put(NS, "m2", {"text": "prefers answers in bullet points", "kind": "preference"})

# another tenant, to prove the boundary rather than assert it
store.put(("globex", "raj", "memories"), "m1", {"text": "globex-only note"})

hits = store.search(("acme",), limit=10)
print("acme prefix sees:", [(h.namespace, h.key) for h in hits])
print("leaked another tenant?", any(h.namespace[0] != "acme" for h in hits))

# put(namespace, key, value, index=None, *, ttl=None)   get(namespace, key)
# search(namespace_prefix, /, *, query=None, filter=None, limit=10, offset=0)   delete(namespace, key)


## Cell 2: The silent failure
Run it and **read the scores**. Without an index, `search(query=...)` returns the entire namespace with every score `None`, and nothing errors.


In [ ]:
# READ THE SCORES. This is the trap this lesson exists to spring.
noindex = InMemoryStore()                      # no index= argument
for k, v in [("m1", "asked about leave encashment limits"),
             ("m2", "prefers answers in bullet points"),
             ("m3", "the cafeteria menu changes on Tuesdays")]:
    noindex.put(NS, k, {"text": v})

hits = noindex.search(NS, query="leave encashment", limit=5)
print("without an index:", [(h.key, h.score) for h in hits])
# -> [('m1', None), ('m2', None), ('m3', None)]
#
# It returned EVERYTHING, including the cafeteria menu, and every score is None. `query=` was
# accepted, no warning was raised, and nothing failed. A store with no embedding index has no
# way to rank - so "semantic memory" silently degrades to "the whole namespace, in insertion
# order", and your prompt fills with irrelevance while looking like it is working.

# Configure the index and the same call ranks:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    vertexai=True, project=PROJECT_ID, location=REGION,   # embeddings are REGIONAL, unlike generation
    output_dimensionality=768)                            # 3072 unless you ask (2.4)

indexed = InMemoryStore(index={"dims": 768, "embed": embeddings, "fields": ["text"]})
for k, v in [("m1", "asked about leave encashment limits"),
             ("m2", "prefers answers in bullet points"),
             ("m3", "the cafeteria menu changes on Tuesdays")]:
    indexed.put(NS, k, {"text": v})

ranked = indexed.search(NS, query="leave encashment", limit=3)
for h in ranked:
    print(f"  {h.key}  score={h.score:.3f}  {h.value['text'][:40]}")

# Assert it, because the failure is silent. `ranked and` matters: all() over an EMPTY list is True.
assert ranked and all(h.score is not None for h in ranked), \
    "no scores - the store has no embedding index and search() is returning everything"

# NOW READ THOSE SCORES AGAIN. m2 and m3 came back too, sorted to the bottom. RANKING IS NOT
# FILTERING, and search() has no score_threshold argument, so "only the relevant ones" is a
# decision you make in Python or do not make at all.
SCORE_FLOOR = 0.15          # tune on YOUR memories - a product decision, not a default


def relevant(store, namespace, query, limit=3, floor=SCORE_FLOOR):
    """Ranked AND filtered. Returns [] when nothing clears the bar, which is a real answer."""
    return [h for h in store.search(namespace, query=query, limit=limit)
            if (h.score or 0) >= floor]


print("above the floor:", [h.key for h in relevant(indexed, NS, "leave encashment")])
print("nothing relevant:", [h.key for h in relevant(indexed, NS, "quarterly revenue in Brazil")])


## Cell 3: Recall at the top of a turn
The recalled facts go into the message list, so they land in the checkpoint - and three weeks later you can see exactly what the model was told.


In [ ]:
from langgraph.config import get_store
from langchain_core.messages import AIMessage, HumanMessage


def last_user_text(state) -> str:
    """The last thing the HUMAN said - not messages[-1], because recall appends to the list."""
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage) or getattr(m, "type", None) == "human":
            return m.content
    return ""


def recall(state, config) -> dict:
    """Fetch what we know about THIS user before the model answers.

    The namespace is built from the thread id, not from a global: one process serves many
    tenants, and the store is only as isolated as the tuple you hand it.
    """
    tenant, user, _ = config["configurable"]["thread_id"].split(":", 2)
    memories = relevant(get_store(), (tenant, user, "memories"), last_user_text(state), limit=3)
    if not memories:
        return {"messages": []}
    lines = "\n".join(f"- {m.value['text']}" for m in memories)
    # Injected as a message the model reads, NOT as a system prompt rewrite: it stays visible in
    # the checkpoint, so when the answer is odd you can see exactly what it was told.
    return {"messages": [AIMessage(content=f"[what I remember about this user]\n{lines}")]}


## Cell 4: Writing a memory is a privileged operation
PII is refused, not redacted; the audit row carries the key, never the text; a refusal is audited too.


In [ ]:
import hashlib
import json as _json
import re
from datetime import datetime, timezone

# Deterministic PII patterns, as constants - never inline in an f-string (lesson 5.5).
PAN_RE = r'\b[A-Z]{5}[0-9]{4}[A-Z]\b'
# Aadhaar never begins 0 or 1, and the separator in the wild is a space or a hyphen.
AADHAAR_RE = r'\b[2-9]\d{3}[\s\-]?\d{4}[\s\-]?\d{4}\b'
# \b will NOT do here: in +919876543210 there is no boundary between the prefix and the number,
# so the anchor has to be "not preceded by a digit" instead.
MOBILE_RE = r'(?<!\d)(?:\+91[\-\s]?)?[6-9]\d{4}[\s\-]?\d{5}(?!\d)'
EMAIL_RE = r'[\w.\-]+@[\w\-]+\.[A-Za-z]{2,}'
# These two OVERLAP: a +91 mobile is twelve digits, which also satisfies the Aadhaar shape. The
# memory is refused either way; the order decides only which label leads in the audit row.
_PII = (("pan", PAN_RE), ("email", EMAIL_RE), ("mobile", MOBILE_RE), ("aadhaar", AADHAAR_RE))

# Positive AND negative cases. A detector nobody has tried to fool is a detector nobody has tested.
assert re.search(PAN_RE, "PAN ABCDE1234F"), "PAN"
assert re.search(MOBILE_RE, "call 98765 43210"), "mobile: split form"
assert re.search(MOBILE_RE, "+919876543210"), "mobile: +91 prefix, no word boundary there"
assert re.search(AADHAAR_RE, "UID 2234 5678 9012"), "aadhaar"
assert not re.search(PAN_RE, "clause NP-03 applies"), "PAN over-match"
assert not re.search(AADHAAR_RE, "policy version 1234 5678 9012"), "aadhaar starts 2-9"
assert {n for n, p in _PII if re.search(p, "+919876543210")} == {"mobile", "aadhaar"}, "the overlap is real - assert it"

# These four are a CHEAP PRE-FILTER, not the control. They miss a lowercase PAN, a passport number,
# a bank account, and anything phrased unusually. Lesson 5.5's argument: a regex finds the
# identifiers you thought of; Sensitive Data Protection finds the ones you did not.


def audit(action: str, tenant: str, user: str, key: str, detail: dict) -> None:
    """One row per memory write or delete. Append-only, and never the memory text itself."""
    row = {"ts": datetime.now(timezone.utc).isoformat(), "action": action,
           "tenant_id": tenant, "user_id": user, "memory_key": key, **detail}
    print("AUDIT", _json.dumps(row))          # production: a row in 12.3's warehouse, like the MCP server's audit line


def remember(store, tenant: str, user: str, text: str, kind: str = "topic") -> str | None:
    """Write a memory, after a PII gate, with an audit row either way. Returns the key, or None if refused."""
    found = [name for name, pat in _PII if re.search(pat, text)]
    key = hashlib.sha256(text.encode()).hexdigest()[:16]      # idempotent: same text, same key
    if found:
        # REFUSED, and the refusal is audited. A memory store is the worst possible home for an
        # identifier: it is retrieved into a prompt on every future conversation.
        audit("refused", tenant, user, key, {"reason": "pii", "kinds": found})
        return None
    store.put((tenant, user, "memories"), key, {"text": text, "kind": kind})
    audit("write", tenant, user, key, {"kind": kind, "chars": len(text)})
    return key


def forget(store, tenant: str, user: str, key: str) -> None:
    """Delete a memory. The audit row is the only thing that survives it."""
    store.delete((tenant, user, "memories"), key)
    audit("delete", tenant, user, key, {"reason": "user_request"})


k1 = remember(indexed, TENANT, USER_ID, "asked about leave encashment limits")
k2 = remember(indexed, TENANT, USER_ID, "contact PAN ABCDE1234F for payroll")
k3 = remember(indexed, TENANT, USER_ID, "reachable on 98765 43210")
print("stored:", k1, "| refused:", k2, k3)

forget(indexed, TENANT, USER_ID, k1)
print("still in the namespace:", sorted(h.key for h in indexed.search(NS, limit=10)))


## Cell 5: interrupt() - stop and ask
A pause that survives the process dying. The node re-runs from its first line on resume: read state above the interrupt, act only below it.


In [ ]:
from langgraph.types import interrupt, Command


def confirm_web_search(state) -> dict:
    """Pause and ask a human before the agent leaves DocuMind's own corpus.

    interrupt() raises a special signal that LangGraph catches: the graph STOPS, the checkpoint
    records exactly where, and invoke() returns with "__interrupt__" in the result. Nothing is
    lost and nothing is running - the process can die here and the decision still resumes.
    """
    decision = interrupt({
        "action": "web_search",
        "query": last_user_text(state)[:120],     # the HUMAN's question, not messages[-1]
        "why": "leaves DocuMind's corpus, bills per search query, and the result is not covered by your retention policy",
    })
    if decision != "approve":
        return {"messages": [AIMessage(content="Web search declined. Answering from DocuMind documents only.")]}
    return {"messages": [AIMessage(content="Web search approved.")]}


# THE NODE MUST BE IDEMPOTENT. interrupt() works by RE-RUNNING the node from the top when you
# resume, so everything above the interrupt() call happens TWICE. Put a side effect there - charge
# a card, send an email, write a row - and it happens twice too. Read state above the interrupt;
# act only below it.


## Cell 6: Wire it together
8.5's graph with `recall` at the front and `confirm` on the branch that leaves the corpus, over the kit's two adapters. `compile()` takes both the checkpointer and the store.


In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool


# THE TOOLS ARE THE KIT'S, behind two adapters - 8.5's retrieve (tenant from the thread id) and the
# six-key cost estimate. Nothing is re-declared here: this lesson once carried its own copy of the
# cost function, and a copy is exactly the drift 8.7 fails the build for.
@tool
def retrieve(query: str, config: RunnableConfig) -> dict:
    """Retrieve grounded passages from DocuMind's document corpus, with the lane's cited answer.

    Args:
        query: The question, in natural language.
    """
    tenant_id = config["configurable"]["thread_id"].split(":", 1)[0]
    return documind_tools.retrieve(query, tenant_id=tenant_id, top_k=5, brain="langgraph")


@tool
def calculate_processing_cost(total_pages: int, num_documents: int = 1, processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents.
        num_documents: How many documents those pages are spread across.
        processing_type: Service tier - standard, priority, or bulk.
    """
    return documind_tools.calculate_processing_cost(total_pages, num_documents, processing_type)


TOOLS = [retrieve, calculate_processing_cost]
SYSTEM = SystemMessage(content="You are DocuMind AI. Use retrieve for any question about the documents and cite the "
                               "sources it returns; use what you remember about this user when it is relevant. "
                               "Pass tenant='acme' to any documind_ tool. Never state a figure a citation does not carry.")

base_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", vertexai=True, project=PROJECT_ID, location="global", thinking_level="low")
llm = base_llm.bind_tools(TOOLS)

# Anything that would send the agent outside DocuMind's own corpus gets a human first.
EXTERNAL = ("gdpr", "regulation", "new law", "news", "benchmark", "competitor")


def agent(state: MessagesState) -> dict:
    # The system prompt is prepended per call, never stored - the checkpoint holds the conversation,
    # not the instructions (deploy/services/chat/brains.py does the same).
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}


def needs_confirmation(state: MessagesState) -> Literal["confirm", "agent"]:
    text = last_user_text(state).lower()          # again: NOT messages[-1]
    return "confirm" if any(w in text for w in EXTERNAL) else "agent"


def after_agent(state: MessagesState) -> Literal["tools", "__end__"]:
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else "__end__"


builder = StateGraph(MessagesState)
builder.add_node("recall", recall)                  # Cell 3
builder.add_node("confirm", confirm_web_search)     # Cell 5
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "recall")                   # remember first, then decide
builder.add_conditional_edges("recall", needs_confirmation)
builder.add_edge("confirm", "agent")
builder.add_conditional_edges("agent", after_agent)
builder.add_edge("tools", "agent")

# BOTH of these, and the second is the line people leave out. checkpointer= gives the graph THIS
# CONVERSATION; store= gives it THIS USER. Compile without store= and get_store() inside recall
# returns None - the first real question dies on an AttributeError that names neither.
app = builder.compile(checkpointer=InMemorySaver(), store=indexed)

for e in app.get_graph().edges:
    print(f"  {e.source:>9} -> {e.target:<9}{'  (conditional)' if e.conditional else ''}")

# THE PAYOFF: a BRAND NEW thread - no history, no checkpoint - that already knows this user, and
# then goes to the corpus for the figure. Watch for the [what I remember] line, then the tool call,
# then the answer with LV-07's encashment rule from ACME's handbook (golden row jn-03's clause).
remember(indexed, TENANT, USER_ID, "asked about leave encashment limits")
FRESH = {"configurable": {"thread_id": f"{TENANT}:{USER_ID}:fresh-1"}}
out = app.invoke({"messages": [{"role": "user", "content": "Remind me what I asked you about before, and tell me "
                                                          "what the handbook says the encashment limit is."}]}, FRESH)
for m in out["messages"]:
    print(f"  {m.type:<9} {str(m.content)[:100]}")
assert any(getattr(m, "tool_calls", None) for m in out["messages"]), "the agent never went to the corpus"


### Resume, both ways
A confirmation step that has only ever been tested with 'approve' is a dialog box, not a control.


In [ ]:
CONFIG = {"configurable": {"thread_id": f"{TENANT}:{USER_ID}:hitl-1"}}
ASK = {"messages": [{"role": "user", "content": "What are the 2026 GDPR fines for late breach notification?"}]}

result = app.invoke(ASK, CONFIG)

# The graph is paused, not finished.
print("interrupted:", "__interrupt__" in result)
print("payload:", result["__interrupt__"][0].value)
print("paused at:", app.get_state(CONFIG).next)          # -> ('confirm',)

# ... hours later, in a different process, after a human clicked a button ...
approved = app.invoke(Command(resume="approve"), CONFIG)

# Print the DECISION and the answer. Printing only messages[-1] shows a plausible reply on both
# branches and tells you nothing about which one ran. Note what the agent does next: the corpus
# holds the DPDP Act and not the GDPR, so an honest answer says the corpus cannot ground it.
for m in approved["messages"][-2:]:
    print(" approve |", str(m.content)[:100])

# And the other branch. A confirm node that has only ever been tested with "yes" is a dialog box.
DENY = {"configurable": {"thread_id": f"{TENANT}:{USER_ID}:hitl-2"}}
app.invoke(ASK, DENY)
declined = app.invoke(Command(resume="deny"), DENY)
for m in declined["messages"][-2:]:
    print(" deny    |", str(m.content)[:100])


## Cell 7: Module 7's deployed server, from this brain
Same server, same IAM, same roster, a different client. The credential is minted per request by an `httpx.Auth`, because this client has no header provider. The tools are bound *and called*.


In [ ]:
import httpx
from langchain_mcp_adapters.client import MultiServerMCPClient


class IdTokenAuth(httpx.Auth):
    """A credential minted per REQUEST for the MCP server, as the roster member this notebook speaks as.

    langchain-mcp-adapters 0.3.2 has no header_provider (ADK's answer, 7.3). Its per-request hook
    is `auth`: an httpx.Auth whose auth_flow runs on every request - the tool list AND every call -
    so no token is ever captured for the life of the client. Checked against the installed package.
    """

    def __init__(self, audience: str):
        self.audience = audience

    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {documind_tools._id_token(self.audience)}"   # the kit's hook
        yield request


# Module 7's DEPLOYED server, from the LangGraph brain. Same server, same IAM, same roster, a
# different client - which is the point of the protocol. The four tools arrive prefixed.
client = MultiServerMCPClient(
    # timeout=120: the adapters' default HTTP timeout is 5 s, like ADK's. A retrieve() through the server
    # is rag-api plus a Gemini answer - up to 90 s when the API is cold - while a roster refusal is
    # instant, so a 5 s client passes the refusal and fails the real question. The first live A2A
    # peer did exactly that; every MCP client in this course now says 120.
    {"documind": {"url": f"{MCP_URL}/mcp", "transport": "streamable_http", "auth": IdTokenAuth(MCP_URL),
                  "timeout": 120, "sse_read_timeout": 300}},
    tool_name_prefix=True,      # -> documind_retrieve, documind_list_documents, documind_corpus_stats, documind_calculate_processing_cost
)
mcp_tools = await client.get_tools()
print("loaded:", [t.name for t in mcp_tools])
assert {"documind_retrieve", "documind_corpus_stats"} <= {t.name for t in mcp_tools}

# Now WIRE them, or they are just a list you printed. `agent` looks `llm` up in the module namespace
# on every call, so rebinding the name is enough - the graph does not need rebuilding. That is
# convenient here and worth being conscious of: in a service you build the tool list once at
# startup and pass it in. tool_name_prefix is not decoration: two servers publishing `search` is
# what happens the second time someone in your company builds one.
llm = base_llm.bind_tools(TOOLS + mcp_tools)
print("agent now holds:", [t.name for t in TOOLS] + [t.name for t in mcp_tools])

# A question only the server can answer - corpus counts - through the same graph, with recall and
# the confirm gate still in front. Golden proof that the wiring is real: a documind_ tool call.
out = app.invoke({"messages": [{"role": "user", "content": "How many chunks and documents does our corpus hold?"}]},
                 {"configurable": {"thread_id": f"{TENANT}:{USER_ID}:mcp-1"}})
calls = [tc["name"] for m in out["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
print("tool calls:", calls)
print("answer    :", str(out["messages"][-1].content)[:200])
assert any(c.startswith("documind_") for c in calls), "the MCP tools were bound but never called"


## Cell 8: Your memory, or Google's


In [ ]:
# THE OTHER ROUTE, and when to take it.
#
# Everything above is memory you own: a store, your namespaces, your PII gate, your audit row.
# Lesson 8.3 showed the managed alternative - deploy an ADK agent to Agent Runtime and it gets
# Sessions and Memory Bank without any of this code:
#
#   remote = client.agent_engines.create(agent=AdkApp(agent=root_agent), config={...})
#   await remote.async_add_session_to_memory(user_id="priya", session_id=session["id"])
#
# Memory Bank extracts durable facts from a conversation for you, which is genuinely more than
# the store above does - that one only remembers what you explicitly write to it.
#
# WHICH ONE:
#   Managed (8.3)  - you want memory and do not want to run a database. It meters through Agent
#                    Compute and Agent Storage; take the number from the live pricing page.
#   Your own       - you need the PII gate to be YOUR policy, the audit row to be in YOUR
#                    warehouse, and "show me everything you stored about this customer" to be a
#                    SELECT. In this course's market, DPDP makes the third one a requirement.
#
# DocuMind runs its own for the LangGraph brain and Memory Bank for the managed ADK agent. The two
# share ONE tool layer, so the memory each keeps is the only thing that differs - and 8.7 asks the
# same question through both.


## Cell 9: The production lane
The same graph, two managed connections - the full profile. The lean lane's chat service runs it on memory, and says so.


In [ ]:
# THE PRODUCTION LANE - the full profile's. Same graph, two managed connections instead of two
# in-memory objects. On the lean profile there is no Cloud SQL: the chat service runs the graph on
# InMemorySaver (CHECKPOINT_DSN=memory) and no store at all, which is the demo's honest limit.
from contextlib import asynccontextmanager
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.store.postgres import AsyncPostgresStore

# Separate strings even when they point at one Cloud SQL instance. The day memories move to
# their own database - a different retention policy, a different DPDP answer - you want that to
# be a config change and not a refactor. Empty here on purpose: this cell DEFINES the factory.
CHECKPOINT_DSN = os.environ.get("CHECKPOINT_DSN", "")
STORE_DSN = os.environ.get("STORE_DSN", "")
INDEX = {"dims": 768, "embed": embeddings, "fields": ["text"]}


@asynccontextmanager
async def production_app():
    """YIELD, do not return: return the graph from inside `async with` and both pools close on the way out."""
    if not CHECKPOINT_DSN or not STORE_DSN or "memory" in (CHECKPOINT_DSN, STORE_DSN):
        raise RuntimeError("no database DSNs - the full profile's cloudsql.tf creates the instance and the secret")
    async with (AsyncPostgresSaver.from_conn_string(CHECKPOINT_DSN) as saver,
                AsyncPostgresStore.from_conn_string(STORE_DSN, index=INDEX) as store):
        yield builder.compile(checkpointer=saver, store=store)     # the SAME builder as Cell 6


# setup() creates the tables, and it runs ONCE - from a migration job, not from the service (8.5's
# rule, for the store as well as the checkpointer). And unlike InMemoryStore, this one supports
# ttl, in MINUTES:  await store.aput(ns, key, {"text": ...}, ttl=90 * 24 * 60)     # 90 days
print("production factory defined; DSNs set:", bool(CHECKPOINT_DSN), bool(STORE_DSN))


## What this adds up to

| Capability | Where it lives | What it must never do |
|---|---|---|
| Recall | a node at the top of the turn | read a namespace built from anything but the thread |
| Write | a gated function | store an identifier, or write with no audit row |
| Confirm | `interrupt()` in its own node | have side effects above the interrupt call |
| MCP tools | a client built at startup, minting per request | hold a one-hour token for the life of the process |

All four are the same rule in different clothes: **the thing that decides must be separate from the thing that acts.** 6.2 argued it about a dispatch seam, 8.5 about a router that runs before the model, 7.1 about a roster check before the API. It keeps returning because it is what makes a system reviewable.

## ✅ Lesson 8.6 complete
- ✅ A store isolated by namespace, an index that ranks, a floor that filters
- ✅ Recall as a message in the checkpoint; writes gated, audited, refusable
- ✅ An interrupt resumed both ways
- ✅ The graph over the kit's two adapters; a fresh thread that remembers and then retrieves
- ✅ The deployed MCP server bound with a credential per request, and called
